In [ ]:
# =========================
# FULL SCRIPT
# 1) Build 100 random DAG datasets (F=37 edges, OF=47 edges)
#    - Each DAG is guaranteed acyclic (DAG)
#    - Each edge has random weight w ~ U(-1, 1)
#    - No subfolders; indexing is in filename
#    - Save ORIGINAL once (train/val/test)
#    - Log DAG validity for each generated DAG
#
# 2) Train/Eval for each DAG (100 runs)
#    - F : edge-only (37 features)
#    - OF: original + edge (orig + 47 features)
#    - Model: LightGBM (GPU try -> CPU fallback)
#    - Model seed fixed (MODEL_SEED constant)
#    - Threshold from VAL by best F1 (grid)
#    - Evaluate on TEST: AUROC, AUPRC, F1, Brier, ECE
#    - Save results separately: results_F.csv, results_OF.csv
#    - Progress/time logs: elapsed, speed, ETA
# =========================

import os
import time
import json
import hashlib
import warnings
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import networkx as nx

import lightgbm as lgb
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")


# -------------------------
# Config
# -------------------------
BASE_DIR = "/content/drive/MyDrive/bank_failure_prediction"

# output directory (single folder)
OUT_DIR = os.path.join(BASE_DIR, "randdag_fixed_edges_100files")
os.makedirs(OUT_DIR, exist_ok=True)

DATA_PREFIX = "data_with_features"
TARGET_COL = "label"

# number of DAGs
N_DAGS = 100

# edge counts (no sorting/sub-selection later)
E_F  = 37
E_OF = 47

# seeds for DAG/weight generation (changes per ds_id)
DAG_SEED_BASE = 7777
WEIGHT_SEED_BASE = 12345

# model/eval
MODEL_SEED = 42
N_ESTIMATORS = 2000
THR_GRID = 101
ECE_BINS = 15

LOG_EVERY = 5  # progress log every N dag runs

# save edge list CSVs (traceability)
SAVE_EDGE_LIST_CSV = True


# -------------------------
# Utilities (naming, logs)
# -------------------------
def fmt_hms(seconds: float) -> str:
    seconds = max(0.0, float(seconds))
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    if h > 0:
        return f"{h:d}h {m:02d}m {s:02d}s"
    return f"{m:d}m {s:02d}s"


# -------------------------
# Hard-block helpers
# -------------------------
def is_indexlike_col_name(c: str) -> bool:
    cl = str(c).strip().lower()
    return cl.startswith("unnamed") or cl in {"index", "_index"} or cl.endswith("_index")


def drop_indexlike_cols(df: pd.DataFrame, name: str) -> pd.DataFrame:
    drop_cols = [c for c in df.columns if is_indexlike_col_name(c)]
    if drop_cols:
        print(f"[DROP] {name}: index-like columns removed -> {drop_cols}")
        df = df.drop(columns=drop_cols)

    bad = [c for c in df.columns if str(c).strip().lower().startswith("unnamed")]
    if bad:
        raise RuntimeError(f"[FATAL] {name}: Unnamed columns still exist: {bad}")
    return df


def fatal_if_unnamed_node(x: str, context: str):
    if "unnamed" in str(x).lower():
        raise RuntimeError(f"[FATAL] {context}: node contains 'Unnamed' -> {x}")


def fatal_if_bad_colname(col: str, context: str):
    if is_indexlike_col_name(col) or ("unnamed" in str(col).strip().lower()):
        raise RuntimeError(f"[FATAL] {context}: forbidden column name -> {col}")


def make_edge_col_name(set_tag: str, ds_tag: str, u: str, v: str) -> str:
    col = f"edge_RANDDAG_{set_tag}_{ds_tag}__{u}__{v}"
    fatal_if_bad_colname(col, f"edge_col_name/{set_tag}/{ds_tag}")
    return col


# -------------------------
# Data loading (original split from existing files)
# -------------------------
def resolve_original_split_path(base_dir: str, split: str) -> str:
    """
    Supports:
      - flat: BASE_DIR/data_with_features_{split}.csv
      - folder: BASE_DIR/**/original/data_with_features_{split}.csv
    """
    fname = f"{DATA_PREFIX}_{split}.csv"
    flat = os.path.join(base_dir, fname)
    if os.path.exists(flat):
        return flat

    for root, dirs, files in os.walk(base_dir):
        if os.path.basename(root) == "original":
            p = os.path.join(root, fname)
            if os.path.exists(p):
                return p

    raise FileNotFoundError(f"[FATAL] original split not found: split={split}")


def load_original_split_from_base(split: str) -> pd.DataFrame:
    p = resolve_original_split_path(BASE_DIR, split)
    df = pd.read_csv(p, low_memory=False)
    df = drop_indexlike_cols(df, f"original_{split}")
    if TARGET_COL not in df.columns:
        raise RuntimeError(f"[FATAL] target '{TARGET_COL}' missing in {p}")
    return df


def load_saved_original_from_outdir(split: str) -> pd.DataFrame:
    p = os.path.join(OUT_DIR, f"{DATA_PREFIX}_{split}.csv")
    if not os.path.exists(p):
        raise FileNotFoundError(f"[FATAL] saved original missing in OUT_DIR: {p}")
    df = pd.read_csv(p, low_memory=False)
    if TARGET_COL not in df.columns:
        raise RuntimeError(f"[FATAL] target '{TARGET_COL}' missing in {p}")
    return df


def load_randdag(split: str, set_tag: str, ds_tag: str) -> pd.DataFrame:
    p = os.path.join(OUT_DIR, f"{DATA_PREFIX}_RANDDAG_{set_tag}_{ds_tag}_{split}.csv")
    if not os.path.exists(p):
        raise FileNotFoundError(f"[FATAL] randdag file missing: {p}")
    df = pd.read_csv(p, low_memory=False)
    if TARGET_COL not in df.columns:
        raise RuntimeError(f"[FATAL] target '{TARGET_COL}' missing in {p}")
    return df


# -------------------------
# Random DAG generation (guaranteed acyclic)
# -------------------------
def generate_random_dag_edges(nodes: List[str], n_edges: int, rng: np.random.Generator) -> List[Tuple[str, str]]:
    """
    Guarantee DAG:
      - sample random topological order
      - only allow edges forward in that order
      - sample unique edges until n_edges
    """
    n = len(nodes)
    if n < 2:
        raise RuntimeError("[FATAL] need at least 2 nodes")

    max_possible = n * (n - 1) // 2
    if n_edges > max_possible:
        raise RuntimeError(f"[FATAL] n_edges={n_edges} exceeds max_possible={max_possible}")

    order = nodes.copy()
    rng.shuffle(order)
    pos = {node: i for i, node in enumerate(order)}

    edges = set()
    while len(edges) < n_edges:
        a = order[int(rng.integers(0, n))]
        b = order[int(rng.integers(0, n))]
        if a == b:
            continue
        if pos[a] < pos[b]:
            edges.add((a, b))
        else:
            edges.add((b, a))

    return list(edges)


def validate_and_log_dag(edges: List[Tuple[str, str]], nodes: List[str], tag: str):
    G = nx.DiGraph()
    G.add_nodes_from(nodes)
    G.add_edges_from(edges)

    is_dag = nx.is_directed_acyclic_graph(G)

    # degree stats
    in_degs = [d for _, d in G.in_degree()]
    out_degs = [d for _, d in G.out_degree()]

    print(f"[DAG] {tag} | nodes={G.number_of_nodes()} edges={G.number_of_edges()} "
          f"| DAG={is_dag} | in(max={max(in_degs)},mean={np.mean(in_degs):.2f}) "
          f"| out(max={max(out_degs)},mean={np.mean(out_degs):.2f})")

    if not is_dag:
        # show a tiny cycle sample, then stop
        cycles = list(nx.simple_cycles(G))
        print(f"[DAG] {tag} cycles example: {cycles[:2]}")
        raise RuntimeError(f"[FATAL] {tag}: generated graph is NOT a DAG")

    # also ensure topo sort works
    _ = list(nx.topological_sort(G))
    return G


# -------------------------
# Metrics
# -------------------------
def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 15) -> float:
    y_true = y_true.astype(int)
    y_prob = np.clip(y_prob, 0.0, 1.0)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (y_prob >= lo) & (y_prob < hi) if i < n_bins - 1 else (y_prob >= lo) & (y_prob <= hi)
        if not np.any(mask):
            continue
        acc = float(y_true[mask].mean())
        conf = float(y_prob[mask].mean())
        ece += (mask.sum() / n) * abs(acc - conf)
    return float(ece)


def brier_score(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = y_true.astype(float)
    y_prob = np.clip(y_prob, 0.0, 1.0)
    return float(np.mean((y_prob - y_true) ** 2))


def best_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray, n_grid: int = 101) -> float:
    thresholds = np.linspace(0.0, 1.0, n_grid)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)


def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> Dict[str, float]:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "AUROC": float(roc_auc_score(y_true, y_prob)),
        "AUPRC": float(average_precision_score(y_true, y_prob)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Brier": brier_score(y_true, y_prob),
        "ECE": expected_calibration_error(y_true, y_prob, n_bins=ECE_BINS),
    }


# -------------------------
# Model (LightGBM GPU try -> CPU fallback)
# -------------------------
LGBM_CPU_BASE = dict(
    n_estimators=N_ESTIMATORS,
    learning_rate=0.02,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    verbose=-1,
    random_state=int(MODEL_SEED),
)
LGBM_GPU_EXTRA = dict(device_type="gpu", gpu_platform_id=0, gpu_device_id=0)

_GPU_LOGGED = False
_GPU_FAIL_REASON = None


def fit_lgbm(X_train, y_train):
    global _GPU_LOGGED, _GPU_FAIL_REASON

    params_gpu = dict(LGBM_CPU_BASE)
    params_gpu.update(LGBM_GPU_EXTRA)

    # GPU try
    try:
        clf = lgb.LGBMClassifier(**params_gpu)
        clf.fit(X_train, y_train)
        if not _GPU_LOGGED:
            print("[INFO] LightGBM GPU enabled (device_type='gpu').")
            _GPU_LOGGED = True
        return clf
    except Exception as e:
        if _GPU_FAIL_REASON is None:
            _GPU_FAIL_REASON = str(e)
            print(f"[WARN] LightGBM GPU failed -> CPU fallback. first_reason={_GPU_FAIL_REASON}")

        clf = lgb.LGBMClassifier(**LGBM_CPU_BASE)
        clf.fit(X_train, y_train)
        return clf


def predict_probs_and_thr(clf, X_val, y_val, X_test):
    val_prob = clf.predict_proba(X_val)[:, 1]
    thr = best_f1_threshold(y_val, val_prob, n_grid=THR_GRID)
    test_prob = clf.predict_proba(X_test)[:, 1]
    return test_prob, thr


# -------------------------
# Part 1) Build datasets
# -------------------------
def build_datasets():
    print("[STEP 1] Build random DAG datasets")
    df_train = load_original_split_from_base("train")
    df_val   = load_original_split_from_base("val")
    df_test  = load_original_split_from_base("test")

    base_cols = [c for c in df_train.columns if c != TARGET_COL and not str(c).startswith("edge_")]
    for c in base_cols:
        fatal_if_unnamed_node(c, "base_cols")

    # save ORIGINAL once into OUT_DIR
    for split, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
        out_p = os.path.join(OUT_DIR, f"{DATA_PREFIX}_{split}.csv")
        if not os.path.exists(out_p):
            df[base_cols + [TARGET_COL]].to_csv(out_p, index=False)
    print(f"[OK] ORIGINAL saved once -> {OUT_DIR}")

    for i in range(N_DAGS):
        ds_tag = f"ds{i:03d}"

        # If all files exist, skip
        need = []
        for split in ["train", "val", "test"]:
            need.append(os.path.join(OUT_DIR, f"{DATA_PREFIX}_RANDDAG_F_{ds_tag}_{split}.csv"))
            need.append(os.path.join(OUT_DIR, f"{DATA_PREFIX}_RANDDAG_OF_{ds_tag}_{split}.csv"))
        if all(os.path.exists(p) for p in need):
            if i == 0:
                print("[SKIP] ds000 already exists (and likely others).")
            continue

        rng_dag_F  = np.random.default_rng(DAG_SEED_BASE + 100000 + i)
        rng_dag_OF = np.random.default_rng(DAG_SEED_BASE + 200000 + i)
        rng_w_F    = np.random.default_rng(WEIGHT_SEED_BASE + 100000 + i)
        rng_w_OF   = np.random.default_rng(WEIGHT_SEED_BASE + 200000 + i)

        edges_F  = generate_random_dag_edges(base_cols, E_F, rng_dag_F)
        edges_OF = generate_random_dag_edges(base_cols, E_OF, rng_dag_OF)

        # DAG validation logs
        validate_and_log_dag(edges_F,  base_cols, f"{ds_tag}/F")
        validate_and_log_dag(edges_OF, base_cols, f"{ds_tag}/OF")

        w_F  = rng_w_F.uniform(-1.0, 1.0, size=len(edges_F)).astype(np.float32)
        w_OF = rng_w_OF.uniform(-1.0, 1.0, size=len(edges_OF)).astype(np.float32)

        if SAVE_EDGE_LIST_CSV:
            pd.DataFrame({"u": [u for u, v in edges_F], "v": [v for u, v in edges_F], "w": w_F.astype(float)}).to_csv(
                os.path.join(OUT_DIR, f"randdag_edges_F_{ds_tag}.csv"), index=False
            )
            pd.DataFrame({"u": [u for u, v in edges_OF], "v": [v for u, v in edges_OF], "w": w_OF.astype(float)}).to_csv(
                os.path.join(OUT_DIR, f"randdag_edges_OF_{ds_tag}.csv"), index=False
            )

        def add_edges(df_src: pd.DataFrame, set_tag: str, edges, weights) -> pd.DataFrame:
            out = df_src[base_cols + [TARGET_COL]].copy()
            X = out[base_cols]
            for (u, v), ww in zip(edges, weights):
                col = make_edge_col_name(set_tag, ds_tag, u, v)
                out[col] = (ww * (X[u].to_numpy(np.float32) * X[v].to_numpy(np.float32))).astype(np.float32)
            return out

        for split, df_src in [("train", df_train), ("val", df_val), ("test", df_test)]:
            outF  = add_edges(df_src, "F",  edges_F,  w_F)
            outOF = add_edges(df_src, "OF", edges_OF, w_OF)

            outF.to_csv(os.path.join(OUT_DIR, f"{DATA_PREFIX}_RANDDAG_F_{ds_tag}_{split}.csv"), index=False)
            outOF.to_csv(os.path.join(OUT_DIR, f"{DATA_PREFIX}_RANDDAG_OF_{ds_tag}_{split}.csv"), index=False)

        if (i + 1) % 10 == 0 or i == 0:
            print(f"[OK] saved {ds_tag} datasets | F_edges={len(edges_F)} OF_edges={len(edges_OF)}")

    print("[STEP 1 DONE] datasets ready.")
    print(" -", OUT_DIR)


# -------------------------
# Part 2) Train/Eval over 100 DAGs
# -------------------------
def train_eval():
    print("\n[STEP 2] Train/Eval over random DAG datasets")
    df_o_train = load_saved_original_from_outdir("train")
    df_o_val   = load_saved_original_from_outdir("val")
    df_o_test  = load_saved_original_from_outdir("test")

    y_train = df_o_train[TARGET_COL].to_numpy(np.int64)
    y_val   = df_o_val[TARGET_COL].to_numpy(np.int64)
    y_test  = df_o_test[TARGET_COL].to_numpy(np.int64)

    orig_cols = [c for c in df_o_train.columns if c != TARGET_COL]
    Xo_tr = df_o_train[orig_cols].to_numpy(np.float32)
    Xo_va = df_o_val[orig_cols].to_numpy(np.float32)
    Xo_te = df_o_test[orig_cols].to_numpy(np.float32)

    rows_F, rows_OF = [], []

    t_global0 = time.time()
    t_block0 = time.time()

    print(f"[RUN] N_DAGS={N_DAGS} | MODEL_SEED fixed={MODEL_SEED} | F_edges={E_F} | OF_edges={E_OF}")

    for i in range(N_DAGS):
        ds_tag = f"ds{i:03d}"

        # ---- Load F/OF datasets
        dfF_tr  = load_randdag("train", "F",  ds_tag)
        dfF_va  = load_randdag("val",   "F",  ds_tag)
        dfF_te  = load_randdag("test",  "F",  ds_tag)

        dfOF_tr = load_randdag("train", "OF", ds_tag)
        dfOF_va = load_randdag("val",   "OF", ds_tag)
        dfOF_te = load_randdag("test",  "OF", ds_tag)

        # detect edge columns (by prefix)
        prefF  = f"edge_RANDDAG_F_{ds_tag}__"
        prefOF = f"edge_RANDDAG_OF_{ds_tag}__"
        edge_cols_F  = [c for c in dfF_tr.columns  if str(c).startswith(prefF)]
        edge_cols_OF = [c for c in dfOF_tr.columns if str(c).startswith(prefOF)]

        if len(edge_cols_F) != E_F:
            raise RuntimeError(f"[FATAL] {ds_tag}/F: edge cols count mismatch. expected={E_F}, got={len(edge_cols_F)}")
        if len(edge_cols_OF) != E_OF:
            raise RuntimeError(f"[FATAL] {ds_tag}/OF: edge cols count mismatch. expected={E_OF}, got={len(edge_cols_OF)}")

        # log which features are used (as requested)
        print(f"[FEATURES] {ds_tag} | F edge cols={len(edge_cols_F)} | OF edge cols={len(edge_cols_OF)}")
        # (너무 길어질 수 있어서 상위 몇 개만)
        print("  F edges sample:", edge_cols_F[:5])
        print("  OF edges sample:", edge_cols_OF[:5])

        # ---- F: edge-only
        t0 = time.time()
        X_tr_F = dfF_tr[edge_cols_F].to_numpy(np.float32)
        X_va_F = dfF_va[edge_cols_F].to_numpy(np.float32)
        X_te_F = dfF_te[edge_cols_F].to_numpy(np.float32)

        clfF = fit_lgbm(X_tr_F, y_train)
        probF, thrF = predict_probs_and_thr(clfF, X_va_F, y_val, X_te_F)
        metF = compute_metrics(y_test, probF, thrF)
        tF = time.time() - t0

        rows_F.append({
            "DAG_ID": i,
            "DAG_TAG": ds_tag,
            "SET": "F",
            "MODEL": "LightGBM",
            "MODEL_SEED": MODEL_SEED,
            "N_FEAT": int(X_tr_F.shape[1]),
            "THRESHOLD": float(thrF),
            "TIME_SEC": float(tF),
            **metF
        })

        # ---- OF: original + edge(47)
        t1 = time.time()
        X_tr_OF = np.concatenate([Xo_tr, dfOF_tr[edge_cols_OF].to_numpy(np.float32)], axis=1)
        X_va_OF = np.concatenate([Xo_va, dfOF_va[edge_cols_OF].to_numpy(np.float32)], axis=1)
        X_te_OF = np.concatenate([Xo_te, dfOF_te[edge_cols_OF].to_numpy(np.float32)], axis=1)

        clfOF = fit_lgbm(X_tr_OF, y_train)
        probOF, thrOF = predict_probs_and_thr(clfOF, X_va_OF, y_val, X_te_OF)
        metOF = compute_metrics(y_test, probOF, thrOF)
        tOF = time.time() - t1

        rows_OF.append({
            "DAG_ID": i,
            "DAG_TAG": ds_tag,
            "SET": "OF",
            "MODEL": "LightGBM",
            "MODEL_SEED": MODEL_SEED,
            "N_FEAT": int(X_tr_OF.shape[1]),
            "THRESHOLD": float(thrOF),
            "TIME_SEC": float(tOF),
            **metOF
        })

        # ---- progress log
        if (i + 1) % LOG_EVERY == 0 or (i + 1) == N_DAGS:
            now = time.time()
            elapsed_total = now - t_global0
            elapsed_block = now - t_block0
            done = i + 1
            left = N_DAGS - done

            sec_per_dag_block = elapsed_block / LOG_EVERY if done % LOG_EVERY == 0 else elapsed_block / max(1, (done % LOG_EVERY))
            per_min = 60.0 / sec_per_dag_block if sec_per_dag_block > 0 else float("inf")
            eta = sec_per_dag_block * left

            print(f"[PROGRESS] {done:3d}/{N_DAGS} | elapsed={fmt_hms(elapsed_total)} | "
                  f"speed={sec_per_dag_block:.2f}s/dag ({per_min:.2f} dag/min) | ETA={fmt_hms(eta)}")
            t_block0 = now

    # save results separately
    out_results_dir = os.path.join(BASE_DIR, "results_randdag_fixed_edges")
    os.makedirs(out_results_dir, exist_ok=True)

    out_F = os.path.join(out_results_dir, f"results_F_{E_F}edges_{N_DAGS}dags.csv")
    out_OF = os.path.join(out_results_dir, f"results_OF_orig_plus_{E_OF}edges_{N_DAGS}dags.csv")

    dfF = pd.DataFrame(rows_F)
    dfOF = pd.DataFrame(rows_OF)

    dfF.to_csv(out_F, index=False)
    dfOF.to_csv(out_OF, index=False)

    print("\n[DONE] saved results:")
    print(" -", out_F)
    print(" -", out_OF)

    # quick summary
    print("\n[MEAN] F (edge-only)")
    print(dfF[["AUROC","AUPRC","F1","Brier","ECE","TIME_SEC"]].mean())

    print("\n[MEAN] OF (orig+edge)")
    print(dfOF[["AUROC","AUPRC","F1","Brier","ECE","TIME_SEC"]].mean())

    return dfF, dfOF


# =========================
# RUN
# =========================
build_datasets()
dfF, dfOF = train_eval()

display(dfF.head(3))
display(dfOF.head(3))
